In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# No-op when you already have the thermo package alongside this notebook (the
# normal case: you cloned the repository and are running from code/chNN/).
# In Colab there is no repository, so fetch the package and the property data.
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# Example: Departure functions from the Peng-Robinson EOS (package version)

Enthalpy and entropy departures from the generalized Peng-Robinson EOS, applied to the isenthalpic (Joule-Thomson) throttling of methane.

> **Package version.** This notebook uses the shared `thermo` package (`PengRobinson`) instead of re-deriving the Peng–Robinson EOS inline. Compare with the self-contained companion [`PR_throttle_CH4_example.ipynb`](PR_throttle_CH4_example.ipynb), which builds every routine from scratch.
>
> Critical constants ($T_c$, $P_c$), the acentric factor $\omega$, and the ideal-gas $C_p$ coefficients are read from `code/data/pure_property.csv` via `PengRobinson.from_database`, so the numbers can differ slightly from the hand-entered SIS Table 6.6-1 values in the self-contained notebook.

## Problem statement

Methane gas undergoes a rapid, adiabatic expansion through a continuous throttling process from upstream conditions $13^\circ$C and 184 bar to a lower pressure at $-43^\circ$C. Calculate the downstream gas pressure.

The throttle is isenthalpic, so in terms of departure functions

$$\Delta\underline{H}^\mathrm{IG} + [\underline{H}-\underline{H}^\mathrm{IG}]_2 - [\underline{H}-\underline{H}^\mathrm{IG}]_1 = 0.$$

## The Peng-Robinson EOS, from the `thermo` package

The generalized Peng-Robinson equation of state (SIS Eq. 6.4-2) is

$$ P = \frac{RT}{\underline{V}-b} - \frac{a(T)}{\underline{V}(\underline{V}+b) + b(\underline{V}-b)} \tag{Eq. 6.4-2}$$

with $b = 0.07780\,RT_c/P_c$ (Eq. 6.7-2), $a(T)=0.45724\,R^2T_c^2/P_c\,\alpha(T)$ (Eq. 6.7-1), $\sqrt{\alpha}=1+\kappa(1-\sqrt{T/T_c})$ (Eq. 6.7-3), and $\kappa = 0.37464 + 1.54226\omega - 0.26992\omega^2$ (Eq. 6.7-4).

All of this — building the cubic, taking its compressibility roots (SIS Table 6.4-3), and the departure functions below — is implemented in `thermo.PengRobinson`. The self-contained companion notebook codes each of these by hand; here we just call them.

The enthalpy and entropy departure functions for the PR EOS (SIS Eqs. 6.4-29, 6.4-30) are

$$[\underline{H} - \underline{H}^\mathrm{IG}] = RT(Z-1) + \frac{T\,da/dT-a}{2\sqrt2\,b}\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right] \tag{Eq. 6.4-29}$$

$$[\underline{S} - \underline{S}^\mathrm{IG}] = R\ln(Z-B) + \frac{da/dT}{2\sqrt2\,b}\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right] \tag{Eq. 6.4-30}$$

and are returned by `pr.departure_H(T, P, phase)` and `pr.departure_S(T, P, phase)` (J/mol and J/(mol·K)). The package uses the correct $R\ln(Z-B)$ in the entropy departure.

In [1]:
import sys; sys.path.append("..")   # so `import thermo` works from the chapter folder
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants, optimize
from thermo import PengRobinson

R = constants.R

In [2]:
# Build methane from the shared property database (Tc, Pc, omega, Cp coefficients)
pr = PengRobinson.from_database('methane')
print(f"{pr.name}: Tc={pr.Tc:.1f} K, Pc={pr.Pc/1e5:.2f} bar, omega={pr.omega:.3f}")
print(f"ideal-gas Cp coefficients (A,B,C,D) = {pr.cp}")

methane: Tc=190.4 K, Pc=46.00 bar, omega=0.011
ideal-gas Cp coefficients (A,B,C,D) = (19.25, 0.05213, 1.197e-05, -1.132e-08)


In [3]:
# Known conditions
T1, P1 = 13 + 273.15, 184e5     # inlet:  13 C, 184 bar
T2      = -43 + 273.15          # outlet temperature: -43 C

# Inlet compressibility (vapor root) and enthalpy departure
Z1 = pr.Z(T1, P1, "vapor")
depH1 = pr.departure_H(T1, P1, "vapor")
print(f"Z1 = {Z1:.3f},  [H-H_IG]_1 = {depH1:.0f} J/mol")

Z1 = 0.771,  [H-H_IG]_1 = -3120 J/mol


## Ideal-gas enthalpy change

The ideal-gas heat capacity follows the Appendix A.II correlation $C_p^*(T)=A+BT+CT^2+DT^3$, so

$$\Delta\underline{H}^\mathrm{IG}=\int_{T_1}^{T_2}C_p^*\,dT = A(T_2-T_1)+\tfrac12 B(T_2^2-T_1^2)+\tfrac13 C(T_2^3-T_1^3)+\tfrac14 D(T_2^4-T_1^4).$$

The coefficients `(A, B, C, D)` are carried on the compound as `pr.cp` (already in J/(mol·K), no unit scaling needed).

In [4]:
# Ideal-gas Cp correlation and its integrals, using the package coefficients pr.cp
def Cp_IG(pr, T):
    A, B, C, D = pr.cp
    return A + B*T + C*T**2 + D*T**3

def dH_IG(pr, T1, T2):
    A, B, C, D = pr.cp
    return (A*(T2-T1) + B*(T2**2-T1**2)/2
            + C*(T2**3-T1**3)/3 + D*(T2**4-T1**4)/4)

def dS_IG(pr, T1, T2, P1, P2):
    A, B, C, D = pr.cp
    return (A*np.log(T2/T1) + B*(T2-T1) + C*(T2**2-T1**2)/2
            + D*(T2**3-T1**3)/3 - R*np.log(P2/P1))

In [5]:
dHIG = dH_IG(pr, T1, T2)
print(f"Delta H_IG = {dHIG:.0f} J/mol")
# Isenthalpic balance -> required outlet departure:
depH2_target = depH1 - dHIG
print(f"[H-H_IG]_2 must equal {depH2_target:.0f} J/mol")

Delta H_IG = -1865 J/mol
[H-H_IG]_2 must equal -1254 J/mol


## Solve for the downstream pressure

Rather than guess-and-check, solve $[\underline{H}-\underline{H}^\mathrm{IG}]_2(T_2,P_2) = [\underline{H}-\underline{H}^\mathrm{IG}]_1 - \Delta\underline{H}^\mathrm{IG}$ for $P_2$ with a root finder.

In [6]:
def residual(P2):
    return pr.departure_H(T2, P2, "vapor") - depH2_target

P2 = optimize.brentq(residual, 1e4, P1)
print(f"Downstream pressure P2 = {P2/1e5:.2f} bar  ({P2:.3e} Pa)")
Z2 = pr.Z(T2, P2, "vapor")
check = dHIG + pr.departure_H(T2, P2, "vapor") - depH1
print(f"Check: Delta H = {check:.1f} J/mol (should be ~0)")

Downstream pressure P2 = 41.46 bar  (4.146e+06 Pa)
Check: Delta H = 0.0 J/mol (should be ~0)


## Entropy generation

The throttle is adiabatic, so the molar entropy change equals the entropy generated:

$$\Delta\underline{S} = \Delta\underline{S}^\mathrm{IG} + [\underline{S}-\underline{S}^\mathrm{IG}]_2 - [\underline{S}-\underline{S}^\mathrm{IG}]_1,\qquad \Delta\underline{S}^\mathrm{IG}=\int_{T_1}^{T_2}\frac{C_p^*}{T}dT - R\ln\frac{P_2}{P_1}.$$

In [7]:
dSIG = dS_IG(pr, T1, T2, P1, P2)
depS1 = pr.departure_S(T1, P1, "vapor")
depS2 = pr.departure_S(T2, P2, "vapor")
dS = dSIG + depS2 - depS1
print(f"Delta S_IG = {dSIG:.2f} J/mol.K")
print(f"Entropy generated = {dS:.2f} J/mol.K  (> 0, as required)")

Delta S_IG = 5.15 J/mol.K
Entropy generated = 9.27 J/mol.K  (> 0, as required)
